# Model 2: TF-IDF + Naive Bayes

Owner: Samuel. Compare counts, TF-IDF, word pairs, smoothing and ComplementNB. Select using repeated CV macro-F1; reserve the fixed test set for the selected configuration only.

## 1. Setup

In [1]:
MEMBER = "Samuel"
MODEL_NAME = "naive_bayes"

In [2]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !git clone https://github.com/michael-alu/nlp_language_technologies.git
    %cd nlp_language_technologies
    %pip install -q -r requirements.txt

if Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.append(os.getcwd())
print("Working directory:", os.getcwd())

Working directory: /home/wiseking/Downloads/nlp_language_technologies


In [3]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.pipeline import Pipeline
from src import config, data_loading, evaluation, plots, experiment_log
from src.cross_validation import cross_validate
from src.reproducibility import set_seed
set_seed(config.RANDOM_SEED)

Random seed set to 42


## 2. Load development data

No test data is used during model selection.

In [4]:
train = data_loading.load_train()
validation = data_loading.load_validation()
train_and_validation = data_loading.load_train_and_validation()
train_ids = data_loading.labels_to_ids(train["category"])
validation_ids = data_loading.labels_to_ids(validation["category"])
display(pd.crosstab(train_and_validation["category"], columns="development support").reindex(config.LABELS))

col_0,development support
category,
kitaifa,1700
michezo,1461
biashara,1155
kimataifa,46
burudani,13


## 3. Build the model

Vocabulary and IDF are learned inside each Pipeline, including independently in each CV training fold. Hold vocabulary settings constant: lowercase word tokens, default two-or-more-character token pattern, min_df=1, max_df=1.0, no stopword list or feature cap. TF-IDF uses raw term frequency, smoothed IDF and L2 normalization. MNB learns empirical class priors; ComplementNB uses norm=False. TF-IDF values are nonnegative, although they are not literal multinomial counts.

In [5]:
def build_model(settings):
    common = dict(lowercase=True, ngram_range=tuple(settings["ngram_range"]), min_df=1, max_df=1.0)
    if settings["representation"] == "count":
        vectorizer = CountVectorizer(**common)
    elif settings["representation"] == "tfidf":
        vectorizer = TfidfVectorizer(**common, norm="l2", smooth_idf=True, sublinear_tf=False)
    else:
        raise ValueError("Unknown representation")
    if settings["classifier"] == "multinomial":
        classifier = MultinomialNB(alpha=settings["alpha"], fit_prior=True)
    elif settings["classifier"] == "complement":
        classifier = ComplementNB(alpha=settings["alpha"], norm=False)
    else:
        raise ValueError("Unknown classifier")
    return Pipeline([("vectorizer", vectorizer), ("classifier", classifier)])

base = dict(representation="count", classifier="multinomial", ngram_range=(1, 1), alpha=1.0)
experiments = [
    ("count_unigrams", base.copy(), "Classic count MNB baseline."),
    ("tfidf_unigrams", {**base, "representation": "tfidf"}, "Does TF-IDF weighting help?"),
    ("tfidf_bigrams", {**base, "representation": "tfidf", "ngram_range": (1, 2)}, "Do short phrases help?"),
]
# Compare the same alpha grid for MNB and CNB: matched pairs isolate the classifier effect.
for classifier in ("multinomial", "complement"):
    for alpha in (0.01, 0.1, 1.0, 10.0):
        if classifier == "multinomial" and alpha == 1.0:
            continue  # Already included as tfidf_bigrams.
        experiments.append((f"{classifier}_bigrams_alpha_{alpha:g}",
                            {**base, "representation": "tfidf", "ngram_range": (1, 2),
                             "classifier": classifier, "alpha": alpha},
                            "Matched TF-IDF bigram smoothing/classifier comparison."))

## 4. Validation and 4b. Cross-validation for every experiment

Use the same 5 folds × 3 repeats (seed 42) for all candidates on train + validation. Each fold builds a fresh pipeline. Log every validation score and CV mean/sample standard deviation, including per-class F1. These fold scores overlap in training data; their SD describes variability, not a confidence interval.

In [6]:
runs = []
for name, settings, notes in experiments:
    print("\nExperiment:", name)
    model = build_model(settings)
    model.fit(train["content"], train_ids)
    assert np.array_equal(model.classes_, np.arange(len(config.LABELS)))
    probabilities = model.predict_proba(validation["content"])
    validation_scores = evaluation.compute_scores(validation_ids, probabilities)
    experiment_log.log_experiment(MEMBER, MODEL_NAME, name, "validation", settings, validation_scores, notes)
    cv_scores = cross_validate(lambda: build_model(settings), train_and_validation)
    experiment_log.log_experiment(MEMBER, MODEL_NAME, name, "cross_validation", settings, cv_scores, notes)
    runs.append(dict(experiment=name, settings=settings.copy(), **cv_scores))

# Stable ordering breaks exact ties in favour of the earlier, simpler candidate.
ranked = pd.DataFrame(runs).sort_values("macro_f1", ascending=False, kind="stable")
best = ranked.iloc[0]
best_settings = best["settings"].copy()
best_experiment = best["experiment"]
display(ranked[["experiment", "macro_f1", "macro_f1_std", "f1_kimataifa", "f1_kimataifa_std", "f1_burudani", "f1_burudani_std", "log_loss"]])
print("Selected using CV only:", best_experiment, best_settings)


Experiment: count_unigrams


Logged experiment 'count_unigrams' to /home/wiseking/Downloads/nlp_language_technologies/results/experiment_logs/naive_bayes.csv


===Fold 1: macro-F1 = 0.5089===


===Fold 2: macro-F1 = 0.5222===


===Fold 3: macro-F1 = 0.5074===


...

===Fold 13: macro-F1 = 0.3758===


===Fold 14: macro-F1 = 0.3790===


===Fold 15: macro-F1 = 0.3888===
Mean macro-F1 = 0.3811 (std 0.0063)
Logged experiment 'complement_bigrams_alpha_10' to /home/wiseking/Downloads/nlp_language_technologies/results/experiment_logs/naive_bayes.csv


,experiment,macro_f1,macro_f1_std,f1_kimataifa,f1_kimataifa_std,f1_burudani,f1_burudani_std,log_loss
6,complement_bigrams_alpha_0.01,0.5325,0.0217,0.0756,0.0963,0.0,0.0,0.6290
3,multinomial_bigrams_alpha_0.01,0.5169,0.0070,0.0000,0.0000,0.0,0.0,0.8828
0,count_unigrams,0.5111,0.0078,0.0000,0.0000,0.0,0.0,3.4674
7,complement_bigrams_alpha_0.1,0.5076,0.0063,0.0000,0.0000,0.0,0.0,0.4955
4,multinomial_bigrams_alpha_0.1,0.4821,0.0086,0.0000,0.0000,0.0,0.0,0.7961
8,complement_bigrams_alpha_1,0.4552,0.0077,0.0000,0.0000,0.0,0.0,0.5222
1,tfidf_unigrams,0.3970,0.0086,0.0000,0.0000,0.0,0.0,0.6760
9,complement_bigrams_alpha_10,0.3811,0.0063,0.0000,0.0000,0.0,0.0,0.8560
2,tfidf_bigrams,0.3798,0.0071,0.0000,0.0000,0.0,0.0,0.7239
5,multinomial_bigrams_alpha_10,0.3168,0.0045,0.0000,0.0000,0.0,0.0,0.7657


Selected using CV only: complement_bigrams_alpha_0.01 {'representation': 'tfidf', 'classifier': 'complement', 'ngram_range': (1, 2), 'alpha': 0.01}


## 5. Final fixed-test evaluation

Fit the selected model on train only, matching the existing logistic regression final protocol. CV used train + validation for selection. Naive Bayes is deterministic on fixed data, so repeated random seeds would give identical results and are not presented as independent runs. Run this section once per completed study.

In [7]:
model = build_model(best_settings)
model.fit(train["content"], train_ids)
test = data_loading.load_test()
test_ids = data_loading.labels_to_ids(test["category"])
test_probabilities = model.predict_proba(test["content"])
test_scores = evaluation.compute_scores(test_ids, test_probabilities)
evaluation.print_classification_report(test_ids, test_probabilities)
experiment_log.log_experiment(MEMBER, MODEL_NAME, "final", "test", best_settings, test_scores,
                              f"Selected by repeated CV: {best_experiment}; final fit on train only.")
display(test_scores)
plots.plot_confusion_matrix(test_ids, test_probabilities, MODEL_NAME, "test")
plots.plot_roc_curves(test_ids, test_probabilities, MODEL_NAME, "test")
evaluation.save_misclassified_examples(test, test_ids, test_probabilities, MODEL_NAME)
zindi_test = data_loading.load_zindi_test()
evaluation.save_zindi_submission(zindi_test, model.predict_proba(zindi_test["content"]), MODEL_NAME)

              precision    recall  f1-score   support

     kitaifa       0.81      0.87      0.84       300
     michezo       0.95      0.94      0.94       258
    biashara       0.83      0.79      0.81       204
   kimataifa       0.00      0.00      0.00         8
    burudani       1.00      0.33      0.50         3

    accuracy                           0.86       773
   macro avg       0.72      0.59      0.62       773
weighted avg       0.85      0.86      0.86       773

Logged experiment 'final' to /home/wiseking/Downloads/nlp_language_technologies/results/experiment_logs/naive_bayes.csv


{'accuracy': 0.8603,
 'log_loss': 0.6539,
 'macro_f1': 0.6182,
 'weighted_f1': 0.8555,
 'f1_kitaifa': 0.8365,
 'f1_michezo': 0.9435,
 'f1_biashara': 0.8111,
 'f1_kimataifa': 0.0,
 'f1_burudani': 0.5}

/home/wiseking/Downloads/nlp_language_technologies/src/plots.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/home/wiseking/Downloads/nlp_language_technologies/src/plots.py:104: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved 108 misclassified examples to /home/wiseking/Downloads/nlp_language_technologies/results/misclassified/naive_bayes.csv


Saved Zindi submission to /home/wiseking/Downloads/nlp_language_technologies/results/submissions/naive_bayes.csv


## 6. Methodology and discussion

See `reports/naive_bayes_methodology.md` for the report subsection and `07_results_comparison.ipynb` for comparisons loaded from all experiment logs. Interpret rare-class F1 alongside support (only 3 entertainment and 8 international test examples). Final five-model conclusions must wait for all models.

References: [Rennie et al. (2003)](https://people.csail.mit.edu/jrennie/papers/icml03-nb.pdf), [Wang and Manning (2012)](https://aclanthology.org/P12-2018/).